# BiLSTM fine-tune hai miền Windows (từ điển mở rộng + replay OpTC)

Fine-tune mô hình B (bỏ field Bro, đầu vào 4 category + delta_t) trên **hai miền Windows** (skrghosh, FiveDirections) với **từ điển mở rộng** và **replay OpTC**; thiết kế ở `02_artifacts/30_finetune_design_final_v1/STEP_30_FINETUNE_DESIGN_FINAL.md`.

**Gắn đúng 2 dataset vào notebook:** (1) `v2_network_dl_kaggle_upload` (OpTC v2, dùng cho replay và đánh giá OpTC TEST); (2) `v3_ft_kaggle_upload` (gói fine-tune, ~25 MB). Bật GPU.

Mỗi fold: dựng mô hình với embedding mở rộng (hàng mới = bản sao `<UNK>`), **kiểm tra khớp zero-shot trước khi train** (không khớp thì dừng), giai đoạn A (3 epoch, chỉ embedding mới + đầu phân loại), giai đoạn B (3 epoch, mở khóa toàn bộ, lr nhỏ), đánh giá một lần trên mẫu test của fold và OpTC TEST. Không chọn ngưỡng/epoch theo dữ liệu mới; ngưỡng cố định: ngưỡng VAL OpTC cũ và 0.5. BiLSTM: lr giai đoạn A 1e-3, giai đoạn B 1e-4, batch 32.

Cảnh báo khi đọc kết quả: chỉ 16 sample dương, 7 đợt tấn công; cửa sổ kề cùng đợt nằm ở fold khác nhau (rò rỉ có chủ ý do chia trộn); nhãn mức tiến trình hoặc suy ra.

In [ ]:
import os, sys, json, shutil
from pathlib import Path
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


## Cấu hình
Một seed (42), 3 fold theo thiết kế. Đổi `FOLDS` để chạy ít hơn (ví dụ `'0'` để thử một fold).

In [ ]:
MODEL = 'bilstm'
FOLDS = '0,1,2'
SEED = 42
OPTC_EVAL = 'test'   # 'none' để bỏ qua đánh giá OpTC TEST (nhanh hơn)
RUN_NAME = f'{MODEL}_finetune_seed{SEED}'
print(MODEL, FOLDS, SEED, RUN_NAME)


## Tìm hai dataset
Gói OpTC v2 được giải nén/liên kết như notebook 06e/07e; gói fine-tune tìm theo `ft_manifest.json`.

In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path


def unwrap_same_name_folder(
    root: Path,
    name: str,
) -> Path:
    path = root / name

    if not path.exists():
        raise FileNotFoundError(
            f"Missing folder: {path}"
        )

    while (path / name).is_dir():
        path = path / name

    return path


def resolve_kaggle_dataset_root(
    *,
    input_root: str | Path = "/kaggle/input",
    working_root: str | Path = "/kaggle/working/optc_network_dl",
) -> Path:
    input_root = Path(input_root)

    manifests = list(
        input_root.rglob(
            "dataset_manifest.json"
        )
    )

    if not manifests:
        raise FileNotFoundError(
            "Cannot find dataset_manifest.json under /kaggle/input"
        )

    if len(manifests) > 1:
        raise RuntimeError(
            "Nhieu dataset_manifest.json duoc tim thay (vd. dataset LOSO co 3 holdout). "
            "Chi gan MOT dataset vao notebook:\n"
            + "\n".join(str(m) for m in manifests)
        )

    source_root = None

    for manifest in manifests:
        candidate = manifest.parent

        if (
            candidate
            / "build_config.json"
        ).exists():
            source_root = candidate
            break

    if source_root is None:
        raise RuntimeError(
            "Found dataset_manifest.json but not build_config.json"
        )

    working_root = Path(working_root)

    if working_root.exists():
        shutil.rmtree(working_root)

    working_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filename in (
        "dataset_manifest.json",
        "build_config.json",
        "README_DATASET.md",
    ):
        src = source_root / filename

        if src.exists():
            (
                working_root / filename
            ).symlink_to(src)

    for folder in (
        "shards",
        "split",
        "preprocessing",
        "integrity",
    ):
        real_folder = unwrap_same_name_folder(
            source_root,
            folder,
        )

        (
            working_root / folder
        ).symlink_to(
            real_folder,
            target_is_directory=True,
        )

    required = [
        "dataset_manifest.json",
        "build_config.json",
        "split/network_sample_manifest.csv",
        "preprocessing/network_category_maps.json",
        "preprocessing/network_numeric_stats.json",
        "preprocessing/network_preprocess_spec.json",
        "preprocessing/network_preprocess_fingerprint.json",
        "shards/train",
        "shards/val",
        "shards/test",
    ]

    for relative in required:
        path = working_root / relative

        print(
            f"{'PASS' if path.exists() else 'FAIL':4s}",
            relative,
        )

        if not path.exists():
            raise FileNotFoundError(path)

    return working_root


DATASET_ROOT = resolve_kaggle_dataset_root()
print('\nDATASET_ROOT =', DATASET_ROOT)
manifest = json.loads((DATASET_ROOT/'dataset_manifest.json').read_text())
print(json.dumps(manifest['split_stats'], indent=2))


In [ ]:
ft_manifests = list(Path('/kaggle/input').rglob('ft_manifest.json'))
assert len(ft_manifests) == 1, ft_manifests
FT_ROOT = ft_manifests[0].parent
print('FT_ROOT =', FT_ROOT)
man = json.loads((FT_ROOT/'ft_manifest.json').read_text())
print(json.dumps(man['summary']['fold_sizes'], indent=1))
print('token mới theo fold:', json.dumps(man['summary']['added_tokens'], ensure_ascii=False)[:1500])


## Ghi mã vào /kaggle/working

In [ ]:
WORK_CODE = Path('/kaggle/working/ft_code')
WORK_CODE.mkdir(parents=True, exist_ok=True)
(WORK_CODE/'05b_train_bilstm.py').write_text('from __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nimport random\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom sklearn.metrics import (\n    average_precision_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_curve,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n)\nfrom torch.nn.utils.rnn import pack_padded_sequence\nfrom torch.utils.data import DataLoader\n\nfrom network_dl_dataset_v1_1 import (\n    PackedNetworkDataset,\n    ShardAwareBatchSampler,\n    collate_sequence_batch,\n)\n\nSCRIPT_VERSION = "step05b-bilstm-baseline-v1.0"\n\n\ndef load_json(path: Path) -> Any:\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef dump_json(path: Path, obj: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef categorical_cardinalities(dataset_root: Path) -> tuple[list[str], list[int]]:\n    spec = load_json(dataset_root / "preprocessing" / "network_preprocess_spec.json")\n    maps = load_json(dataset_root / "preprocessing" / "network_category_maps.json")\n    fields = list(spec["active_categorical_features"])\n    cards = []\n    for field in fields:\n        mapping = maps["fields"][field]["mapping"]\n        cards.append(max(int(v) for v in mapping.values()) + 1)\n    return fields, cards\n\n\ndef numeric_fields(dataset_root: Path) -> list[str]:\n    spec = load_json(dataset_root / "preprocessing" / "network_preprocess_spec.json")\n    return list(spec["active_numerical_features"])\n\n\ndef embedding_dim(cardinality: int) -> int:\n    return int(min(16, max(4, round(math.sqrt(cardinality) * 2))))\n\n\nclass BiLSTMBaseline(nn.Module):\n    def __init__(\n        self,\n        *,\n        categorical_cardinalities: list[int],\n        numerical_dim: int,\n        binary_dim: int = 1,\n        event_dim: int = 64,\n        hidden_dim: int = 64,\n        lstm_layers: int = 1,\n        dropout: float = 0.2,\n    ):\n        super().__init__()\n        self.embeddings = nn.ModuleList([\n            nn.Embedding(cardinality, embedding_dim(cardinality))\n            for cardinality in categorical_cardinalities\n        ])\n        cat_dim = sum(emb.embedding_dim for emb in self.embeddings)\n        raw_event_dim = cat_dim + numerical_dim + numerical_dim + binary_dim + 1\n\n        self.event_encoder = nn.Sequential(\n            nn.Linear(raw_event_dim, event_dim),\n            nn.LayerNorm(event_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n        )\n        self.lstm = nn.LSTM(\n            input_size=event_dim,\n            hidden_size=hidden_dim,\n            num_layers=lstm_layers,\n            batch_first=True,\n            bidirectional=True,\n            dropout=dropout if lstm_layers > 1 else 0.0,\n        )\n        self.classifier = nn.Sequential(\n            nn.LayerNorm(hidden_dim * 2),\n            nn.Linear(hidden_dim * 2, hidden_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(hidden_dim, 1),\n        )\n\n    def forward(\n        self,\n        *,\n        x_cat: torch.Tensor,\n        x_num: torch.Tensor,\n        x_num_missing: torch.Tensor,\n        x_bin: torch.Tensor,\n        delta_t_log1p: torch.Tensor,\n        lengths: torch.Tensor,\n    ) -> torch.Tensor:\n        cat_parts = [embedding(x_cat[:, :, i]) for i, embedding in enumerate(self.embeddings)]\n        event = torch.cat([*cat_parts, x_num, x_num_missing, x_bin, delta_t_log1p], dim=-1)\n        event = self.event_encoder(event)\n        packed = pack_padded_sequence(event, lengths.cpu(), batch_first=True, enforce_sorted=False)\n        _, (h_n, _) = self.lstm(packed)\n        pooled = torch.cat([h_n[-2], h_n[-1]], dim=-1)\n        return self.classifier(pooled).squeeze(-1)\n\n\ndef make_loader(dataset, *, batch_size: int, shuffle: bool, seed: int, num_workers: int, pin_memory: bool):\n    sampler = ShardAwareBatchSampler(\n        dataset,\n        batch_size,\n        shuffle=shuffle,\n        seed=seed,\n        drop_last=False,\n    )\n    loader = DataLoader(\n        dataset,\n        batch_sampler=sampler,\n        num_workers=num_workers,\n        pin_memory=pin_memory,\n        persistent_workers=num_workers > 0,\n        collate_fn=collate_sequence_batch,\n    )\n    return loader, sampler\n\n\ndef move_batch(batch: dict[str, Any], device: torch.device) -> dict[str, Any]:\n    result = dict(batch)\n    for key in ("x_cat", "x_num", "x_num_missing", "x_bin", "delta_t_log1p", "labels", "lengths"):\n        result[key] = batch[key].to(device, non_blocking=True)\n    return result\n\n\ndef binary_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> dict[str, Any]:\n    y_pred = (y_prob >= threshold).astype(np.uint8)\n    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()\n    result = {\n        "threshold": float(threshold),\n        "precision": float(precision_score(y_true, y_pred, zero_division=0)),\n        "recall": float(recall_score(y_true, y_pred, zero_division=0)),\n        "f1": float(f1_score(y_true, y_pred, zero_division=0)),\n        "fpr": float(fp / (fp + tn) if (fp + tn) else 0.0),\n        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),\n        "pr_auc": float(average_precision_score(y_true, y_prob)),\n        "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else None,\n    }\n    return result\n\n\ndef best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> tuple[float, dict[str, Any]]:\n    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)\n    if len(thresholds) == 0:\n        return 0.5, binary_metrics(y_true, y_prob, 0.5)\n    p, r = precision[:-1], recall[:-1]\n    denom = p + r\n    f1 = np.divide(2 * p * r, denom, out=np.zeros_like(denom), where=denom > 0)\n    index = int(np.nanargmax(f1))\n    threshold = float(thresholds[index])\n    return threshold, binary_metrics(y_true, y_prob, threshold)\n\n\n@torch.no_grad()\ndef predict(model, loader, *, device, amp: bool, max_batches: int | None = None):\n    model.eval()\n    probs, labels, sample_ids = [], [], []\n    for step, raw_batch in enumerate(loader):\n        if max_batches is not None and step >= max_batches:\n            break\n        batch = move_batch(raw_batch, device)\n        with torch.autocast(\n            device_type=device.type,\n            dtype=torch.float16,\n            enabled=amp and device.type == "cuda",\n        ):\n            logits = model(\n                x_cat=batch["x_cat"],\n                x_num=batch["x_num"],\n                x_num_missing=batch["x_num_missing"],\n                x_bin=batch["x_bin"],\n                delta_t_log1p=batch["delta_t_log1p"],\n                lengths=batch["lengths"],\n            )\n        probs.append(torch.sigmoid(logits).cpu().numpy())\n        labels.append(batch["labels"].cpu().numpy())\n        sample_ids.extend(raw_batch["sample_ids"])\n    return np.concatenate(labels), np.concatenate(probs), sample_ids\n\n\ndef save_predictions(path: Path, *, sample_ids, y_true, y_prob, threshold: float) -> None:\n    y_pred = (y_prob >= threshold).astype(np.uint8)\n    with path.open("w", newline="", encoding="utf-8") as f:\n        w = csv.writer(f)\n        w.writerow(["sample_id", "label", "probability", "prediction"])\n        for sid, y, p, pred in zip(sample_ids, y_true, y_prob, y_pred):\n            w.writerow([sid, int(y), float(p), int(pred)])\n\n\ndef save_curves(output_dir: Path, *, y_true, y_prob, prefix: str) -> None:\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay\n\n    fig, ax = plt.subplots()\n    PrecisionRecallDisplay.from_predictions(y_true, y_prob, ax=ax)\n    ax.set_title(f"{prefix.upper()} Precision-Recall")\n    fig.tight_layout()\n    fig.savefig(output_dir / f"{prefix}_pr_curve.png", dpi=150)\n    plt.close(fig)\n\n    if len(np.unique(y_true)) > 1:\n        fig, ax = plt.subplots()\n        RocCurveDisplay.from_predictions(y_true, y_prob, ax=ax)\n        ax.set_title(f"{prefix.upper()} ROC")\n        fig.tight_layout()\n        fig.savefig(output_dir / f"{prefix}_roc_curve.png", dpi=150)\n        plt.close(fig)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--dataset-root", required=True)\n    p.add_argument("--output-dir", required=True)\n    p.add_argument("--epochs", type=int, default=8)\n    p.add_argument("--batch-size", type=int, default=64)\n    p.add_argument("--event-dim", type=int, default=64)\n    p.add_argument("--hidden-dim", type=int, default=64)\n    p.add_argument("--lstm-layers", type=int, default=1)\n    p.add_argument("--dropout", type=float, default=0.2)\n    p.add_argument("--lr", type=float, default=1e-3)\n    p.add_argument("--weight-decay", type=float, default=1e-4)\n    p.add_argument("--grad-clip", type=float, default=1.0)\n    p.add_argument("--patience", type=int, default=3)\n    p.add_argument("--num-workers", type=int, default=2)\n    p.add_argument("--cache-shards", type=int, default=1)\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--no-amp", action="store_true")\n    p.add_argument("--max-train-batches", type=int, default=None, help="Debug only")\n    p.add_argument("--max-eval-batches", type=int, default=None, help="Debug only")\n    args = p.parse_args()\n\n    dataset_root = Path(args.dataset_root)\n    output_dir = Path(args.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    seed_everything(args.seed)\n\n    manifest = load_json(dataset_root / "dataset_manifest.json")\n    train_stats = manifest["split_stats"]["train"]\n    train_positive = int(train_stats["positive_samples"])\n    train_negative = int(train_stats["negative_samples"])\n    if train_positive <= 0:\n        raise RuntimeError("TRAIN has no positive samples")\n    pos_weight_value = train_negative / train_positive\n\n    cat_fields, cat_cards = categorical_cardinalities(dataset_root)\n    num_fields = numeric_fields(dataset_root)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    amp = (not args.no_amp) and device.type == "cuda"\n\n    print("=" * 100)\n    print("STEP 05B — BiLSTM baseline")\n    print("=" * 100)\n    print(f"[DATASET] {dataset_root}")\n    print(f"[TRAIN] negative={train_negative:,} | positive={train_positive:,} | pos_weight={pos_weight_value:.6f}")\n    print(f"[CAT] {list(zip(cat_fields, cat_cards))}")\n    print(f"[NUM] {num_fields}")\n    print(f"[DEVICE] {device} | amp={amp}")\n\n    train_ds = PackedNetworkDataset(dataset_root, "train", cache_shards=args.cache_shards)\n    val_ds = PackedNetworkDataset(dataset_root, "val", cache_shards=args.cache_shards)\n    test_ds = PackedNetworkDataset(dataset_root, "test", cache_shards=args.cache_shards)\n    pin = device.type == "cuda"\n\n    train_loader, train_sampler = make_loader(train_ds, batch_size=args.batch_size, shuffle=True, seed=args.seed, num_workers=args.num_workers, pin_memory=pin)\n    val_loader, _ = make_loader(val_ds, batch_size=args.batch_size, shuffle=False, seed=args.seed, num_workers=args.num_workers, pin_memory=pin)\n    test_loader, _ = make_loader(test_ds, batch_size=args.batch_size, shuffle=False, seed=args.seed, num_workers=args.num_workers, pin_memory=pin)\n\n    model = BiLSTMBaseline(\n        categorical_cardinalities=cat_cards,\n        numerical_dim=len(num_fields),\n        event_dim=args.event_dim,\n        hidden_dim=args.hidden_dim,\n        lstm_layers=args.lstm_layers,\n        dropout=args.dropout,\n    ).to(device)\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    print(f"[PARAMS] {trainable:,}")\n\n    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, dtype=torch.float32, device=device))\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n\n    config = {\n        "script_version": SCRIPT_VERSION,\n        "dataset_root": str(dataset_root),\n        "dataset_preprocess_fingerprint": (manifest["source"].get("step04c_preprocess_fingerprint") or manifest["source"]["step05f_c_preprocess_fingerprint"]),\n        "seed": args.seed,\n        "epochs": args.epochs,\n        "batch_size": args.batch_size,\n        "event_dim": args.event_dim,\n        "hidden_dim": args.hidden_dim,\n        "lstm_layers": args.lstm_layers,\n        "dropout": args.dropout,\n        "lr": args.lr,\n        "weight_decay": args.weight_decay,\n        "grad_clip": args.grad_clip,\n        "patience": args.patience,\n        "num_workers": args.num_workers,\n        "amp": amp,\n        "loss": "BCEWithLogitsLoss",\n        "train_pos_weight": pos_weight_value,\n        "threshold_policy": "maximize F1 on VAL only",\n        "early_stopping_metric": "VAL PR-AUC",\n        "test_policy": "evaluate once after best epoch + VAL threshold are frozen",\n        "categorical_fields": cat_fields,\n        "categorical_cardinalities": cat_cards,\n        "numerical_fields": num_fields,\n        "trainable_parameters": trainable,\n    }\n    dump_json(output_dir / "config.json", config)\n\n    best_pr_auc = -1.0\n    best_epoch = -1\n    stale_epochs = 0\n    history = []\n    best_path = output_dir / "best_model.pt"\n\n    for epoch in range(1, args.epochs + 1):\n        train_sampler.set_epoch(epoch)\n        model.train()\n        running_loss = 0.0\n        seen_batches = 0\n        started = time.time()\n\n        for step, raw_batch in enumerate(train_loader, start=1):\n            if args.max_train_batches is not None and step > args.max_train_batches:\n                break\n            batch = move_batch(raw_batch, device)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):\n                logits = model(\n                    x_cat=batch["x_cat"],\n                    x_num=batch["x_num"],\n                    x_num_missing=batch["x_num_missing"],\n                    x_bin=batch["x_bin"],\n                    delta_t_log1p=batch["delta_t_log1p"],\n                    lengths=batch["lengths"],\n                )\n                loss = criterion(logits, batch["labels"])\n\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)\n            scaler.step(optimizer)\n            scaler.update()\n\n            running_loss += float(loss.item())\n            seen_batches += 1\n            if step % 200 == 0:\n                print(f"[EPOCH {epoch}] batch={step}/{len(train_loader)} loss={running_loss/seen_batches:.6f}")\n\n        train_loss = running_loss / max(1, seen_batches)\n        val_true, val_prob, _ = predict(model, val_loader, device=device, amp=amp, max_batches=args.max_eval_batches)\n        val_pr_auc = float(average_precision_score(val_true, val_prob))\n        val_threshold, val_metrics = best_f1_threshold(val_true, val_prob)\n        elapsed = time.time() - started\n\n        row = {\n            "epoch": epoch,\n            "train_loss": train_loss,\n            "val_pr_auc": val_pr_auc,\n            "val_best_f1_threshold": val_threshold,\n            "val_precision": val_metrics["precision"],\n            "val_recall": val_metrics["recall"],\n            "val_f1": val_metrics["f1"],\n            "val_fpr": val_metrics["fpr"],\n            "elapsed_seconds": elapsed,\n        }\n        history.append(row)\n        print(f"[EPOCH {epoch}] loss={train_loss:.6f} | val_PR_AUC={val_pr_auc:.6f} | val_F1={val_metrics[\'f1\']:.6f} | thr={val_threshold:.6f} | {elapsed/60:.1f} min")\n\n        if val_pr_auc > best_pr_auc + 1e-8:\n            best_pr_auc = val_pr_auc\n            best_epoch = epoch\n            stale_epochs = 0\n            torch.save({\n                "model_state_dict": model.state_dict(),\n                "config": config,\n                "best_epoch": best_epoch,\n                "best_val_pr_auc": best_pr_auc,\n            }, best_path)\n            print(f"[CHECKPOINT] epoch={epoch}")\n        else:\n            stale_epochs += 1\n\n        if stale_epochs >= args.patience:\n            print(f"[EARLY STOP] no VAL PR-AUC improvement for {stale_epochs} epochs")\n            break\n\n    if not history or not best_path.exists():\n        raise RuntimeError("Training did not produce a best checkpoint")\n\n    with (output_dir / "history.csv").open("w", newline="", encoding="utf-8") as f:\n        w = csv.DictWriter(f, fieldnames=list(history[0].keys()))\n        w.writeheader()\n        w.writerows(history)\n\n    checkpoint = torch.load(best_path, map_location=device, weights_only=False)\n    model.load_state_dict(checkpoint["model_state_dict"])\n\n    val_true, val_prob, val_ids = predict(model, val_loader, device=device, amp=amp, max_batches=args.max_eval_batches)\n    threshold, val_metrics = best_f1_threshold(val_true, val_prob)\n    threshold_payload = {\n        "selected_on": "val",\n        "policy": "maximize F1",\n        "threshold": threshold,\n        "best_epoch": best_epoch,\n        "best_val_pr_auc": best_pr_auc,\n        "val_metrics_at_threshold": val_metrics,\n    }\n    dump_json(output_dir / "threshold.json", threshold_payload)\n    save_predictions(output_dir / "val_predictions.csv", sample_ids=val_ids, y_true=val_true, y_prob=val_prob, threshold=threshold)\n    save_curves(output_dir, y_true=val_true, y_prob=val_prob, prefix="val")\n\n    # TEST is touched only after checkpoint and threshold are frozen on VAL.\n    test_true, test_prob, test_ids = predict(model, test_loader, device=device, amp=amp, max_batches=args.max_eval_batches)\n    test_metrics = binary_metrics(test_true, test_prob, threshold)\n    save_predictions(output_dir / "test_predictions.csv", sample_ids=test_ids, y_true=test_true, y_prob=test_prob, threshold=threshold)\n    save_curves(output_dir, y_true=test_true, y_prob=test_prob, prefix="test")\n\n    final_metrics = {\n        "script_version": SCRIPT_VERSION,\n        "best_epoch": best_epoch,\n        "best_val_pr_auc": best_pr_auc,\n        "threshold_selected_on_val": threshold,\n        "val": val_metrics,\n        "test": test_metrics,\n    }\n    dump_json(output_dir / "metrics.json", final_metrics)\n\n    print("\\n" + "=" * 100)\n    print("[SUCCESS] BiLSTM baseline complete")\n    print(f"[BEST EPOCH] {best_epoch}")\n    print(f"[VAL]  PR-AUC={val_metrics[\'pr_auc\']:.6f} | F1={val_metrics[\'f1\']:.6f} | Recall={val_metrics[\'recall\']:.6f} | FPR={val_metrics[\'fpr\']:.6f}")\n    print(f"[TEST] PR-AUC={test_metrics[\'pr_auc\']:.6f} | F1={test_metrics[\'f1\']:.6f} | Recall={test_metrics[\'recall\']:.6f} | FPR={test_metrics[\'fpr\']:.6f}")\n    print(f"[OUTPUT] {output_dir}")\n    print("=" * 100)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(WORK_CODE/'network_dl_dataset_v1_1.py').write_text('from __future__ import annotations\n\nimport csv\nimport json\nimport random\nfrom collections import OrderedDict, defaultdict\nfrom pathlib import Path\nfrom typing import Any, Iterator\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset, Sampler\n\n\nclass PackedNetworkDataset(Dataset):\n    def __init__(self, dataset_root: str | Path, split: str, *, cache_shards: int = 2):\n        self.dataset_root = Path(dataset_root)\n        self.split = split.lower().strip()\n        self.cache_shards = max(1, int(cache_shards))\n        self.app_family_col = None\n        self.cat_idx = None\n        self.drop_numeric = False\n        self.zero_binary = False\n        subset_file = self.dataset_root / "preprocessing" / "feature_subset.json"\n        if subset_file.exists():\n            cfg = json.loads(subset_file.read_text(encoding="utf-8"))\n            order = cfg["x_cat_columns_all"]\n            self.cat_idx = [order.index(f) for f in cfg["keep_categorical"]]\n            self.drop_numeric = bool(cfg["drop_numeric"])\n            self.zero_binary = bool(cfg["zero_binary"])\n\n        if self.split not in {"train", "val", "test"}:\n            raise ValueError("split must be train/val/test")\n\n        manifest = self.dataset_root / "split" / "network_sample_manifest.csv"\n        if not manifest.exists():\n            raise FileNotFoundError(manifest)\n\n        self.rows: list[dict[str, Any]] = []\n        with manifest.open("r", newline="", encoding="utf-8") as f:\n            for row in csv.DictReader(f):\n                if row["split"] != self.split:\n                    continue\n                self.rows.append({\n                    "sample_id": row["sample_id"],\n                    "label": int(row["label"]),\n                    "event_count": int(row["event_count"]),\n                    "shard_file": row["shard_file"],\n                    "shard_sample_index": int(row["shard_sample_index"]),\n                })\n\n        if not self.rows:\n            raise RuntimeError(f"No samples found for split={self.split}")\n\n        self._cache: OrderedDict[str, dict[str, np.ndarray]] = OrderedDict()\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def _load_shard(self, relative_path: str) -> dict[str, np.ndarray]:\n        cached = self._cache.get(relative_path)\n        if cached is not None:\n            self._cache.move_to_end(relative_path)\n            return cached\n\n        path = self.dataset_root / relative_path\n        if not path.exists():\n            raise FileNotFoundError(path)\n\n        with np.load(path, allow_pickle=False) as z:\n            shard = {key: z[key] for key in z.files}\n\n        self._cache[relative_path] = shard\n        self._cache.move_to_end(relative_path)\n        while len(self._cache) > self.cache_shards:\n            self._cache.popitem(last=False)\n        return shard\n\n    def _x_cat(self, x):\n        if self.cat_idx is None:\n            return x\n        return x[:, self.cat_idx]  # fancy indexing tao ban sao, shard cache giu nguyen\n\n    def _x_num(self, x):\n        return x[:, :0] if self.drop_numeric else x\n\n    def _x_bin(self, x):\n        return np.zeros_like(x) if self.zero_binary else x\n\n    def __getitem__(self, index: int) -> dict[str, Any]:\n        row = self.rows[index]\n        shard = self._load_shard(row["shard_file"])\n        i = row["shard_sample_index"]\n        offsets = shard["sample_offsets"]\n        start = int(offsets[i])\n        end = int(offsets[i + 1])\n\n        if end - start != row["event_count"]:\n            raise RuntimeError(f"Manifest/shard event_count mismatch: {row[\'sample_id\']}")\n\n        return {\n            "sample_id": row["sample_id"],\n            "label": int(shard["sample_labels"][i]),\n            "event_count": end - start,\n            "x_cat": self._x_cat(shard["x_cat"][start:end]),\n            "x_num": self._x_num(shard["x_num"][start:end]),\n            "x_num_missing": self._x_num(shard["x_num_missing"][start:end]),\n            "x_bin": self._x_bin(shard["x_bin"][start:end]),\n            "delta_t_log1p": shard["delta_t_log1p"][start:end],\n        }\n\n\nclass ShardAwareBatchSampler(Sampler[list[int]]):\n    def __init__(self, dataset: PackedNetworkDataset, batch_size: int, *, shuffle: bool, seed: int = 42, drop_last: bool = False):\n        self.dataset = dataset\n        self.batch_size = int(batch_size)\n        self.shuffle = bool(shuffle)\n        self.seed = int(seed)\n        self.drop_last = bool(drop_last)\n        self.epoch = 0\n        if self.batch_size <= 0:\n            raise ValueError("batch_size must be > 0")\n\n        grouped: dict[str, list[int]] = defaultdict(list)\n        for idx, row in enumerate(dataset.rows):\n            grouped[row["shard_file"]].append(idx)\n        self.groups = list(grouped.items())\n\n        total = 0\n        for _, indices in self.groups:\n            total += len(indices) // self.batch_size if self.drop_last else (len(indices) + self.batch_size - 1) // self.batch_size\n        self._len = total\n\n    def set_epoch(self, epoch: int) -> None:\n        self.epoch = int(epoch)\n\n    def __len__(self) -> int:\n        return self._len\n\n    def __iter__(self) -> Iterator[list[int]]:\n        rng = random.Random(self.seed + self.epoch)\n        groups = [(shard, list(indices)) for shard, indices in self.groups]\n        if self.shuffle:\n            rng.shuffle(groups)\n\n        for _, indices in groups:\n            if self.shuffle:\n                rng.shuffle(indices)\n            for start in range(0, len(indices), self.batch_size):\n                batch = indices[start:start + self.batch_size]\n                if self.drop_last and len(batch) < self.batch_size:\n                    continue\n                yield batch\n\n\ndef collate_sequence_batch(batch: list[dict[str, Any]]) -> dict[str, Any]:\n    if not batch:\n        raise ValueError("Empty batch")\n\n    lengths = torch.tensor([x["event_count"] for x in batch], dtype=torch.long)\n    max_len = int(lengths.max().item())\n    bsz = len(batch)\n    c = batch[0]["x_cat"].shape[1]\n    n = batch[0]["x_num"].shape[1]\n    binary_dim = batch[0]["x_bin"].shape[1]\n\n    x_cat = torch.zeros(bsz, max_len, c, dtype=torch.long)\n    x_num = torch.zeros(bsz, max_len, n, dtype=torch.float32)\n    x_missing = torch.zeros(bsz, max_len, n, dtype=torch.float32)\n    x_bin = torch.zeros(bsz, max_len, binary_dim, dtype=torch.float32)\n    delta = torch.zeros(bsz, max_len, 1, dtype=torch.float32)\n    mask = torch.zeros(bsz, max_len, dtype=torch.bool)\n    labels = torch.tensor([x["label"] for x in batch], dtype=torch.float32)\n\n    for i, item in enumerate(batch):\n        t = item["event_count"]\n        x_cat[i, :t] = torch.from_numpy(np.asarray(item["x_cat"], dtype=np.int64))\n        x_num[i, :t] = torch.from_numpy(np.asarray(item["x_num"], dtype=np.float32))\n        x_missing[i, :t] = torch.from_numpy(np.asarray(item["x_num_missing"], dtype=np.float32))\n        x_bin[i, :t] = torch.from_numpy(np.asarray(item["x_bin"], dtype=np.float32))\n        delta[i, :t, 0] = torch.from_numpy(np.asarray(item["delta_t_log1p"], dtype=np.float32))\n        mask[i, :t] = True\n\n    return {\n        "x_cat": x_cat,\n        "x_num": x_num,\n        "x_num_missing": x_missing,\n        "x_bin": x_bin,\n        "delta_t_log1p": delta,\n        "attention_mask": mask,\n        "labels": labels,\n        "lengths": lengths,\n        "sample_ids": [x["sample_id"] for x in batch],\n    }\n', encoding='utf-8')
(WORK_CODE/'s14_ft_train.py').write_text('#!/usr/bin/env python3\n"""S14b - Fine-tune mô hình B (BiLSTM hoặc Mixed-Type) trên hai miền Windows (skrghosh, FiveDirections) với từ điển mở rộng và replay OpTC (STEP 30). Chạy trên Kaggle; chạy thử CPU được.\n\nMỗi fold k:\n  1. dựng mô hình với bảng embedding mở rộng (hàng mới = bản sao hàng <UNK>); các tham số khác copy từ mô hình gốc; kiến trúc không đổi.\n  2. KIỂM TRA BẮT BUỘC: trước khi train, logit trên mẫu test của fold phải trùng dự đoán zero-shot tham chiếu (STEP 26, 29) trong dung sai; không qua thì dừng.\n  3. giai đoạn A: chỉ học hàng embedding mới + đầu phân loại; giai đoạn B: mở khóa toàn bộ, lr nhỏ. Mỗi epoch = toàn bộ mẫu TRAIN của hai miền (fold != k) + một lần rút mới từ tập replay OpTC.\n  4. đánh giá MỘT lần: mẫu test của fold (hai miền) và OpTC TEST (tuỳ chọn); ngưỡng cố định: ngưỡng VAL OpTC của mô hình gốc và 0.5. Không chọn ngưỡng, epoch hay siêu tham số theo dữ liệu mới.\nĐầu vào mô hình: 4 category (app_family, proto, resp_port_class, image_dir_class) + delta_t_log1p; numeric rỗng; x_bin = 0 (giống bản B)."""\nimport argparse\nimport importlib.util\nimport json\nimport random\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\nCAT_IDX = [0, 1, 4, 5]  # app_family, proto, resp_port_class, image_dir_class trong x_cat 7 cột của shards\nCAT_NAMES = ["app_family", "proto", "resp_port_class", "image_dir_class"]\nEMB_PREFIX = {"bilstm": "embeddings.", "mixed": "event_encoder.cat_embeddings."}\n\n\ndef load_module(path, name):\n    sys.path.insert(0, str(Path(path).parent))  # 05b_train_bilstm.py import network_dl_dataset_v1_1 cùng thư mục\n    spec = importlib.util.spec_from_file_location(name, path)\n    mod = importlib.util.module_from_spec(spec)\n    sys.modules[name] = mod\n    spec.loader.exec_module(mod)\n    return mod\n\n\n# ---------------------------------------------------------------- dữ liệu\ndef read_shard(path):\n    with np.load(path, allow_pickle=False) as z:\n        return {k: z[k] for k in ("x_cat", "delta_t_log1p", "sample_offsets")}\n\n\ndef sample_from(arrs, i, label, sid):\n    off = arrs["sample_offsets"]\n    a, b = int(off[i]), int(off[i + 1])\n    return {"x_cat": arrs["x_cat"][a:b][:, CAT_IDX].astype(np.int64), "delta": arrs["delta_t_log1p"][a:b].astype(np.float32), "label": int(label), "sample_id": sid}\n\n\ndef materialize(rows):\n    """rows: DataFrame có cột root, shard_file, shard_sample_index, label, sample_id -> list sample (giữ thứ tự rows)."""\n    out = [None] * len(rows)\n    rows = rows.reset_index(drop=True)\n    for (root, shard), g in rows.groupby(["root", "shard_file"], sort=False):\n        arrs = read_shard(Path(root) / shard)\n        for pos, r in zip(g.index, g.itertuples()):\n            out[pos] = sample_from(arrs, int(r.shard_sample_index), r.label, r.sample_id)\n    return out\n\n\ndef collate(batch):\n    lengths = torch.tensor([len(s["delta"]) for s in batch], dtype=torch.long)\n    T, B = int(lengths.max()), len(batch)\n    x_cat = torch.zeros(B, T, len(CAT_IDX), dtype=torch.long)\n    delta = torch.zeros(B, T, 1)\n    mask = torch.zeros(B, T, dtype=torch.bool)\n    for i, s in enumerate(batch):\n        t = len(s["delta"])\n        x_cat[i, :t] = torch.from_numpy(s["x_cat"])\n        delta[i, :t, 0] = torch.from_numpy(s["delta"])\n        mask[i, :t] = True\n    return {"x_cat": x_cat, "x_num": torch.zeros(B, T, 0), "x_num_missing": torch.zeros(B, T, 0), "x_bin": torch.zeros(B, T, 1), "delta_t_log1p": delta, "mask": mask,\n            "lengths": lengths, "labels": torch.tensor([s["label"] for s in batch], dtype=torch.float32), "sample_ids": [s["sample_id"] for s in batch]}\n\n\ndef forward(model, kind, b, device):\n    kw = {k: b[k].to(device) for k in ("x_cat", "x_num", "x_num_missing", "x_bin", "delta_t_log1p")}\n    if kind == "bilstm":\n        return model(lengths=b["lengths"], **kw)\n    return model(attention_mask=b["mask"].to(device), **kw)\n\n\n@torch.no_grad()\ndef predict(model, kind, samples, device, bs):\n    model.eval()\n    logit, labels, ids = [], [], []\n    for i in range(0, len(samples), bs):\n        b = collate(samples[i:i + bs])\n        logit.append(forward(model, kind, b, device).float().cpu().numpy())\n        labels.append(b["labels"].numpy())\n        ids += b["sample_ids"]\n    logit = np.concatenate(logit) if logit else np.zeros(0)\n    prob = torch.sigmoid(torch.from_numpy(logit).half()).float().numpy()  # giống mô phỏng fp16 ở STEP 26\n    return pd.DataFrame({"sample_id": ids, "label": (np.concatenate(labels) if labels else np.zeros(0)).astype(int), "logit": logit, "probability": prob})\n\n\ndef metrics_at(y, p, thr):\n    pred = (p >= thr).astype(int)\n    tp = int(((y == 1) & (pred == 1)).sum()); fp = int(((y == 0) & (pred == 1)).sum()); fn = int(((y == 1) & (pred == 0)).sum()); tn = int(((y == 0) & (pred == 0)).sum())\n    pr = tp / (tp + fp) if tp + fp else 0.0\n    rc = tp / (tp + fn) if tp + fn else 0.0\n    f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0\n    return {"threshold": float(thr), "tp": tp, "fp": fp, "fn": fn, "tn": tn, "precision": round(pr, 4), "recall": round(rc, 4), "f1": round(f1, 4)}\n\n\ndef summarize(df, thresholds):\n    y, p, lg = df.label.to_numpy(), df.probability.to_numpy(), df.logit.to_numpy()\n    out = {"n": int(len(df)), "positives": int(y.sum()), "at": {name: metrics_at(y, p, t) for name, t in thresholds.items()}}\n    if 0 < y.sum() < len(y):\n        out["pr_auc"] = round(float(average_precision_score(y, lg)), 4)\n        out["roc_auc"] = round(float(roc_auc_score(y, lg)), 4)\n    return out\n\n\n# ---------------------------------------------------------------- mô hình\ndef build_model(kind, cfg, cards, code_dir):\n    if kind == "bilstm":\n        m = load_module(Path(code_dir) / "05b_train_bilstm.py", "train_bilstm")\n        base_dims = [m.embedding_dim(c) for c in cfg["categorical_cardinalities"]]  # giữ nguyên chiều embedding của mô hình gốc dù số hàng tăng (hàm gốc suy chiều từ số hàng)\n        it = iter(base_dims)\n        m.embedding_dim = lambda c: next(it)\n        return m.BiLSTMBaseline(categorical_cardinalities=cards, numerical_dim=0, event_dim=cfg["event_dim"], hidden_dim=cfg["hidden_dim"], lstm_layers=cfg["lstm_layers"], dropout=cfg["dropout"])\n    m = load_module(Path(code_dir) / "mixed_type_attention_model.py", "mixed_model")\n    return m.MixedTypeSelfAttentionClassifier(m.MixedTypeModelConfig(categorical_cardinalities=cards, numerical_dim=0, binary_dim=1, max_events=cfg["max_events"], d_model=cfg["d_model"],\n                                                                      num_heads=cfg["num_heads"], feature_layers=cfg["feature_layers"], temporal_layers=cfg["temporal_layers"], ff_dim=cfg["ff_dim"],\n                                                                      dropout=cfg["dropout"]))\n\n\ndef transfer_weights(model, base_sd, old_cards, kind):\n    """Copy tham số gốc; hàng embedding mới = bản sao hàng <UNK> (id 0). Trả về {tên tham số embedding: số hàng cũ}."""\n    sd = model.state_dict()\n    new_sd, old_rows = {}, {}\n    for key, tensor in sd.items():\n        if key not in base_sd:\n            raise KeyError(f"tham số {key} không có trong mô hình gốc")\n        base = base_sd[key]\n        if key.startswith(EMB_PREFIX[kind]) and key.endswith(".weight"):\n            idx = int(key[len(EMB_PREFIX[kind]):].split(".")[0])\n            assert base.shape[1] == tensor.shape[1], f"chiều embedding đổi ở {key}: {tuple(base.shape)} -> {tuple(tensor.shape)}"\n            assert base.shape[0] == old_cards[idx] and tensor.shape[0] >= base.shape[0]\n            w = tensor.clone()\n            w[:base.shape[0]] = base\n            w[base.shape[0]:] = base[0:1]\n            new_sd[key] = w\n            old_rows[key] = base.shape[0]\n        else:\n            assert base.shape == tensor.shape, (key, base.shape, tensor.shape)\n            new_sd[key] = base\n    model.load_state_dict(new_sd)\n    return old_rows\n\n\ndef set_stage(model, kind, stage, old_rows):\n    """A: chỉ hàng embedding mới và đầu phân loại. B: tất cả. Trả về danh sách hook cần gỡ."""\n    handles = []\n    for name, p in model.named_parameters():\n        if stage == "B":\n            p.requires_grad = True\n        elif name in old_rows:\n            p.requires_grad = True\n            mask = torch.zeros(p.shape[0], 1, device=p.device)\n            mask[old_rows[name]:] = 1.0\n            handles.append(p.register_hook(lambda g, m=mask: g * m))\n        else:\n            p.requires_grad = name.startswith("classifier.")\n    return handles\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\n# ---------------------------------------------------------------- chạy một fold\ndef run_fold(args, k, ctx):\n    dev = ctx["device"]\n    kind = args.model\n    ft = Path(args.ft_root)\n    fold_dir = ft / f"fold{k}"\n    spec = json.load(open(fold_dir / "spec_ext.json", encoding="utf-8"))\n    cards = [max(e["id"] for e in spec["categorical"]["fields"][f]["vocab"]) + 1 for f in CAT_NAMES]\n    cfg, old_cards = ctx["cfg"], ctx["cfg"]["categorical_cardinalities"]\n    print(f"[fold {k}] cardinalities cũ {old_cards} -> mở rộng {cards}", flush=True)\n    seed_everything(args.seed + k)\n    model = build_model(kind, cfg, cards, args.code_dir)\n    old_rows = transfer_weights(model, ctx["base_sd"], old_cards, kind)\n    model.to(dev)\n    # dữ liệu hai miền\n    folds = pd.read_csv(ft / "folds.csv")\n    rows = []\n    for ds in ctx["domains"]:\n        sm = pd.read_csv(fold_dir / ds / "data/samples.csv")[["sample_id", "shard_file", "shard_sample_index"]]\n        m = sm.merge(folds[folds.dataset == ds][["sample_id", "fold", "label"]], on="sample_id")\n        m["root"] = str(fold_dir / ds)\n        m["dataset"] = ds\n        rows.append(m)\n    rows = pd.concat(rows, ignore_index=True)\n    samples = materialize(rows)\n    tr_idx = [i for i in range(len(rows)) if rows.fold.iat[i] != k]\n    te_idx = [i for i in range(len(rows)) if rows.fold.iat[i] == k]\n    train_new = [samples[i] for i in tr_idx]\n    test_samples = [samples[i] for i in te_idx]\n    print(f"[fold {k}] train {len(train_new)} mẫu (dương {sum(s[\'label\'] for s in train_new)}), test {len(test_samples)} (dương {sum(s[\'label\'] for s in test_samples)})", flush=True)\n    thr = {"optc_val_threshold": ctx["thr"], "0.5": 0.5}\n    # kiểm tra khớp zero-shot\n    pre = predict(model, kind, test_samples, dev, args.eval_batch)\n    pre["dataset"] = rows.dataset.iloc[te_idx].to_numpy()\n    worst = 0.0\n    for ds in ctx["domains"]:\n        ref = pd.read_csv(ft / "ref_zeroshot" / f"predictions_{ds}_{kind}.csv")[["sample_id", "logit"]].rename(columns={"logit": "ref_logit"})\n        m = pre[pre.dataset == ds].merge(ref, on="sample_id")\n        assert len(m) == (pre.dataset == ds).sum(), f"thiếu mẫu tham chiếu {ds}"\n        d = float((m.logit - m.ref_logit).abs().max())\n        worst = max(worst, d)\n        print(f"[fold {k}] khớp zero-shot {ds}: chênh logit lớn nhất {d:.5f} trên {len(m)} mẫu", flush=True)\n    if worst > args.equiv_tol:\n        raise RuntimeError(f"mô hình mở rộng chưa train KHÔNG khớp zero-shot (chênh {worst:.5f} > {args.equiv_tol}); dừng")\n    result = {"fold": k, "cardinalities_old": old_cards, "cardinalities_ext": cards, "equivalence_max_logit_diff": worst,\n              "zero_shot_on_test": summarize(pre, thr), "added_tokens": spec["extension"]["added"], "history": []}\n    # replay\n    rep = ctx["replay"]\n    rep_pos = [s for s in rep if s["label"] == 1]\n    rep_neg = [s for s in rep if s["label"] == 0]\n    n_pos = sum(s["label"] for s in train_new) + args.replay_pos\n    n_neg = len(train_new) - sum(s["label"] for s in train_new) + (args.replay_per_epoch - args.replay_pos)\n    pos_weight = torch.tensor(n_neg / max(n_pos, 1), dtype=torch.float32, device=dev)\n    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n    amp = dev.type == "cuda" and not args.no_amp\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n    rng = np.random.default_rng(args.seed * 1000 + k)\n    ep_global = 0\n    for stage, epochs, lr, wd in (("A", args.epochs_a, args.lr_a, 0.0), ("B", args.epochs_b, args.lr_b_bilstm if kind == "bilstm" else args.lr_b_mixed, args.weight_decay)):\n        handles = set_stage(model, kind, stage, old_rows)\n        params = [p for p in model.parameters() if p.requires_grad]\n        opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)\n        print(f"[fold {k}] giai đoạn {stage}: {epochs} epoch, lr {lr}, tham số học {sum(p.numel() for p in params):,}", flush=True)\n        for ep in range(epochs):\n            ep_global += 1\n            draw = [rep_pos[i] for i in rng.permutation(len(rep_pos))[:args.replay_pos]] + [rep_neg[i] for i in rng.permutation(len(rep_neg))[:args.replay_per_epoch - args.replay_pos]]\n            epoch_samples = train_new + draw\n            order = rng.permutation(len(epoch_samples))\n            model.train()\n            tot, nb, t0 = 0.0, 0, time.time()\n            for i in range(0, len(order), args.batch_size):\n                b = collate([epoch_samples[j] for j in order[i:i + args.batch_size]])\n                opt.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=amp):\n                    loss = crit(forward(model, kind, b, dev).float(), b["labels"].to(dev))\n                scaler.scale(loss).backward()\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(params, args.grad_clip)\n                scaler.step(opt)\n                scaler.update()\n                tot += float(loss.item())\n                nb += 1\n            result["history"].append({"stage": stage, "epoch": ep_global, "train_loss": round(tot / max(nb, 1), 5), "seconds": round(time.time() - t0, 1)})\n            print(f"[fold {k}] {stage} epoch {ep_global}: loss {tot / max(nb, 1):.5f} ({time.time() - t0:.0f}s)", flush=True)\n        for h in handles:\n            h.remove()\n    # đánh giá một lần\n    post = predict(model, kind, test_samples, dev, args.eval_batch)\n    post["dataset"] = rows.dataset.iloc[te_idx].to_numpy()\n    post["fold"] = k\n    result["after_on_test"] = summarize(post, thr)\n    result["after_on_test_by_dataset"] = {ds: summarize(post[post.dataset == ds], thr) for ds in ctx["domains"]}\n    result["before_on_test_by_dataset"] = {ds: summarize(pre[pre.dataset == ds], thr) for ds in ctx["domains"]}\n    out = Path(args.out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    post.to_csv(out / f"predictions_{kind}_fold{k}_newdomains.csv", index=False)\n    torch.save({"model_state_dict": model.state_dict(), "cardinalities": cards, "fold": k}, out / f"model_{kind}_fold{k}.pt")\n    if args.optc_eval != "none":\n        result["optc_test"] = eval_optc(model, kind, args, ctx, thr, k)\n    json.dump(result, open(out / f"result_{kind}_fold{k}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)\n    print(f"[fold {k}] xong; test sau fine-tune:", json.dumps(result["after_on_test"]["at"], ensure_ascii=False), flush=True)\n    return result\n\n\ndef eval_optc(model, kind, args, ctx, thr, k):\n    """OpTC TEST (đọc streaming theo shard); với mô hình mở rộng id cũ < 61 nên dùng nguyên shards OpTC."""\n    t0 = time.time()\n    man = ctx["optc_manifest"]\n    te = man[man.split == "test"]\n    if args.max_optc_test:\n        te = pd.concat([te[te.label == 1], te[te.label == 0].head(args.max_optc_test)])\n    root = Path(args.optc_root)\n    frames = []\n    for shard, g in te.groupby("shard_file", sort=False):\n        arrs = read_shard(root / shard)\n        samples = [sample_from(arrs, int(r.shard_sample_index), r.label, r.sample_id) for r in g.itertuples()]\n        frames.append(predict(model, kind, samples, ctx["device"], args.eval_batch))\n    df = pd.concat(frames, ignore_index=True)\n    df.to_csv(Path(args.out_dir) / f"predictions_{kind}_fold{k}_optc_test.csv", index=False)\n    res = summarize(df, thr)\n    res["seconds"] = round(time.time() - t0)\n    print(f"[fold {k}] OpTC TEST: {json.dumps(res[\'at\'], ensure_ascii=False)} ({res[\'seconds\']}s)", flush=True)\n    return res\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model", choices=["bilstm", "mixed"], required=True)\n    ap.add_argument("--ft-root", required=True, help="gói v3_ft_kaggle_upload (có ft_manifest.json)")\n    ap.add_argument("--optc-root", required=True, help="gốc dataset OpTC v2 (split/, shards/)")\n    ap.add_argument("--code-dir", required=True, help="thư mục chứa 05b_train_bilstm.py và mixed_type_attention_model.py")\n    ap.add_argument("--out-dir", required=True)\n    ap.add_argument("--folds", default="0,1,2")\n    ap.add_argument("--seed", type=int, default=42)\n    ap.add_argument("--epochs-a", type=int, default=3)\n    ap.add_argument("--epochs-b", type=int, default=3)\n    ap.add_argument("--lr-a", type=float, default=1e-3)\n    ap.add_argument("--lr-b-bilstm", type=float, default=1e-4)\n    ap.add_argument("--lr-b-mixed", type=float, default=3e-5)\n    ap.add_argument("--weight-decay", type=float, default=1e-4)\n    ap.add_argument("--grad-clip", type=float, default=1.0)\n    ap.add_argument("--replay-per-epoch", type=int, default=600)\n    ap.add_argument("--replay-pos", type=int, default=150)\n    ap.add_argument("--batch-size", type=int, default=0, help="0 = mặc định theo mô hình (BiLSTM 32, Mixed 16)")\n    ap.add_argument("--eval-batch", type=int, default=0)\n    ap.add_argument("--equiv-tol", type=float, default=0.02)\n    ap.add_argument("--optc-eval", choices=["test", "none"], default="test")\n    ap.add_argument("--max-optc-test", type=int, default=0, help="chỉ để chạy thử: giới hạn số mẫu âm OpTC TEST")\n    ap.add_argument("--no-amp", action="store_true")\n    ap.add_argument("--cpu", action="store_true")\n    args = ap.parse_args()\n    args.batch_size = args.batch_size or (32 if args.model == "bilstm" else 16)\n    args.eval_batch = args.eval_batch or (128 if args.model == "bilstm" else 32)\n    dev = torch.device("cpu" if args.cpu or not torch.cuda.is_available() else "cuda")\n    ft = Path(args.ft_root)\n    man = json.load(open(ft / "ft_manifest.json", encoding="utf-8"))\n    base = ft / "base_models" / args.model\n    cfg = json.load(open(base / "config.json", encoding="utf-8"))\n    thr = float(json.load(open(base / "threshold.json", encoding="utf-8"))["threshold"])\n    ck = torch.load(base / "best_model.pt", map_location="cpu", weights_only=False)\n    print(f"[init] mô hình {args.model}, thiết bị {dev}, ngưỡng VAL OpTC {thr}, thư mục ra {args.out_dir}", flush=True)\n    optc_manifest = pd.read_csv(Path(args.optc_root) / "split/network_sample_manifest.csv")\n    # tập replay (đọc một lần)\n    rr = pd.read_csv(ft / "replay_R.csv")\n    rr["root"] = str(Path(args.optc_root))\n    t0 = time.time()\n    replay = materialize(rr)\n    print(f"[init] replay R: {len(replay)} mẫu (dương {sum(s[\'label\'] for s in replay)}) nạp trong {time.time() - t0:.0f}s", flush=True)\n    ctx = {"device": dev, "cfg": cfg, "base_sd": ck["model_state_dict"], "thr": thr, "domains": man["domains"], "replay": replay, "optc_manifest": optc_manifest}\n    results = []\n    for k in [int(x) for x in args.folds.split(",")]:\n        results.append(run_fold(args, k, ctx))\n    json.dump({"model": args.model, "args": vars(args), "folds": [r["fold"] for r in results]}, open(Path(args.out_dir) / f"run_{args.model}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)\n    print("[xong]", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Wrote:', sorted(p.name for p in WORK_CODE.iterdir()))
sys.path.insert(0, str(WORK_CODE))


## Chạy fine-tune

In [ ]:
OUTPUT_DIR = Path(f'/kaggle/working/{RUN_NAME}')
cmd = (
    f"python {WORK_CODE/'s14_ft_train.py'} --model {MODEL} "
    f"--ft-root '{FT_ROOT}' --optc-root '{DATASET_ROOT}' --code-dir '{WORK_CODE}' --out-dir '{OUTPUT_DIR}' "
    f"--folds {FOLDS} --seed {SEED} --optc-eval {OPTC_EVAL}"
)
print(cmd)
import subprocess
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'exit code {ret}')


## Kết quả từng fold

In [ ]:
for p in sorted(OUTPUT_DIR.glob('result_*.json')):
    r = json.loads(p.read_text())
    print('=== fold', r['fold'], '| khớp zero-shot (chênh logit lớn nhất):', r['equivalence_max_logit_diff'])
    print('  zero-shot trên test fold :', json.dumps(r['zero_shot_on_test']['at'], ensure_ascii=False))
    print('  sau fine-tune trên test  :', json.dumps(r['after_on_test']['at'], ensure_ascii=False))
    print('  AUC sau fine-tune        :', r['after_on_test'].get('pr_auc'), r['after_on_test'].get('roc_auc'))
    if 'optc_test' in r:
        print('  OpTC TEST                :', json.dumps(r['optc_test']['at'], ensure_ascii=False))


## Nén kết quả để tải về

In [ ]:
archive = shutil.make_archive(f'/kaggle/working/{RUN_NAME}', 'zip', OUTPUT_DIR)
print('Created:', archive)
